# Does the protein agree with the pathology report?

Breast tumors are tested for estrogen receptor (ER) in the clinic. TCGA *also* measured ER-alpha protein directly. **Do the two agree, for which tumors, and does the answer change depending on which samples you include?**

That's the kind of question every product in this challenge has to handle: real data from different sources, joined by IDs, where each join changes who is in the cohort.

*Research only: no diagnosis, treatment advice or clinical claims.*

## 1 · Get the data
This notebook works from a single downloaded file. The first cell verifies its bundled helpers and manifests, then fetches and checks the required TCGA-BRCA tables directly from UCSC Xena. It does not fetch from GitHub. In Colab use **File → Upload notebook**, then **Run all**.

In [ ]:
# @title Prepare the pinned starter files
# Bundled by starter/build_portable.py. No GitHub access is used at runtime.
import base64, gzip, hashlib, pathlib, subprocess, sys, tempfile

BUNDLE = {
    'fetch.py': ('221518f99a06fb6205337ce81fc7f38df5c328aa5f73744c8a433f5d351df75e',
        (
            'H4sIAAAAAAAC/41U32/aMBB+z19x8x5INDC02jqJlQdKu/Vl1cRabdKEIpNciEewM9tpS6vub9/ZoR2g7kdewOe78/d994Mx'
            'dqpvVKVFDq5EuJx8GPdOppMxFLJCC5W0DnOQCqxuTIaWZ/aajk5DLpzou2whenOTib4RN30QKodrNLJYA4qshKzEbGmbFY+i'
            'KysWCHFh9Cq8ZLDWYLR2yRCgXrtSq/2UBbqs5PUatr6XgPQAuUu1gPjn2wF8PEki+Nv3j+S9ntI9o0RIbpeyDvCOXlNimF60'
            'QkTvgxrCIIQ4kuSJSN3MK5nB1eTzBL4i5SmbeReUdmCdNq14BNcGxlaSbc0jxlgkV7U2DkjQLpTClpWkuFq49o9d2y40pqID'
            'N/ijQeui6PxsegajRyf+iX7jNPUI0zQhN6ura4wTXhNS5aLp+At5h6A+MKoQi6IcC7ClOHxzFPs0yTCIV5LfBgPf3Laq3khX'
            'gq5RBe8uZZmzBISFYvikeqENVbpRS09VOjRxJVbzXAyhIEwijw/g+BgOB0kX5owlw51ylbypqTQYhwztowZdYxRdlXibywVR'
            'JzQB+UpIFW8yEDu+WubSxHhLbZrq5ejSNNim8IVMfVVHwB4rzEIfry0XZnEdvGqj5xWuLHkNokcqRt94R6oLP5WZmxIDohQ0'
            'eJRyaxhYskVIFr8f9rNAqb4xXx4249YJ46zXM2Z0n+4LURsaq7hgoQUB7n1sx8d2Zg9st8UzrZxUDT4Zc5KIOPh697cf3Qbm'
            'O9L78aCWjRMgrptae3sCL0ZtbGtks+fxhQHYwwfxvb8Mpvnaoe3MEkJygEdDflA8+Bndo7Db2ZyOVHQjabjjFkNQOCU7m3UD'
            '7mRH5W3co//CrZfh+CddsbK4H7hpjlcjOHg25eR8fPHh7HQ35zBshRY/iQ6VVgs01Ll+b9hwSSyV61i/Q+5Q+Y1ppVYc2DOL'
            'jF1iVYUobRZCyTtyfge5Vh0HjUWat9C035t8QbuGVkBTOcs3xHyzU7kdTSCJ9kTIc4UBjRQZ01SJFa0PryJLUz9gacpaJdpp'
            'i34BWB9UIyUGAAA='
        ),
    ),
    'sources.csv': ('c1a5045d30ab40ba51889c38c20b5dd610f1f4230ec9c52fb35e388b05756a55',
        (
            'H4sIAAAAAAAC/62US2/bOBSF9/0V3AzgArLC92O6SlxMJosEhVvMVrgkL2MBsuSKUhr31w/lVJjO2pFWOiTvOfxE3tR2WHVw'
            'xrHKwzwGbOaxq/IBuNKVP0+Yq64N2GesRpzGFl8wfghd27cBunrKL9X6QTbMSRKGbj72+WN1mKZT/vPmZgrPsH3FHraH2ddZ'
            '1HPeIuRpy2o4ws+hhx+5DsPxJg4/+m6AePNtd39b3+13t3WG46nDRzj9wf9ahGb1eoQS5bUSDr3wSJO3GjUoY3zkIVCmKAeT'
            'QAE3DMsbPHAn0cWgpYTonLI0mlQx7axkolosyXDCnkAImDPZjBjbXEz8PLVDTyYcj5lMQ9lfn9rx+LHilOstY1tGP+R5fGlf'
            'VhzzCBNGsopkcym+Wyl9hgnIHt9gX4NpNVjZ/Jfidap0cMgsoAvcW182JW2ICjQNEZ1Bay0F5GglShuTFAaoMNYzg1o5b0Sl'
            'jWPuSi7HeYJlVvOMPTZdOToXRPXzz2pRtheFrLNK8ceduIbIMYjfrFYu/1cXOkuAgJA0UJaisyJgCiYFsF7xpINWjLOyYWu8'
            'UUlGZqh3KcRADSLTIGw5MkpIydSVhE7jMGHbN+PpBCuaXxq5xC319l++3L7/dVqqNvu7p8XRWC7AgDBKOpUMqsSYjDQYLAiM'
            'QGpo8A6CpM5YlZylUkXGwRvGnEJdGUmNkVeyCMPp3PTz0ePYPJdVbViJLCPkbYT4M1l+JtncP3z99rDjZDqMmA9DFzG+P6X7'
            'Sw7e7EqCp7doqwRd1/zmXfvz5ZTlS2CpPEsWgblgY+DUR70QtVqApKE8WgjFbKQQqSoty6SUCnllLMOCt+JCS2evvX9jD82h'
            'zfh9Bbl/uiX4eiqh87J489CVZt32QP5uv+L3T0RL8nj3/hQv1f/hSwKuhU/acKlKS/KOaq6NVuUG0pSooDpRKw0DrgFSVIrK'
            '0paQReCcCWGkrrTkhaIwV6L5F1ljHfL3BgAA'
        ),
    ),
    'sample_manifest.csv': ('49b62c7805517a04c4719cb84d6dbfbd8e6e14b1564fdc0b0f0aa9a11a099945',
        (
            'H4sIAAAAAAAC/63d2a7rNrbu8fvzFPUACWC1li7VWrLlTm4k+ybIrloHZwFpCmk2UG9/NG2npvnPStWa3/DN2psB+KtBiuQg'
            'aU371+9//OcPn777/I9v/vn9b58//fTb2//76/0//vavf3765vNP3/39h88/ff779z988/++//W7X3//5X8//++j8OPvv021'
            'fv7pVvj7z//813c//f7j/3z65Vb+5afvb//3n7/8/Nunzz998+sPn//x6dfv/u/Pv3x3/x/4P8dikX0bFN9mWXb6duZ981z+'
            'ZvfL5x+//+Vffzv+/uPPv3xz/OX3T1/8p/7+h18/feM9WV3rWl37MSt4tpawlgZrBWv1Mcu/W2Hz1j/v1qMsWVE31T3O/m09'
            'ygbLg+Vp1vHbLN1n79a9LFmZ9202O+T/th5l6Tne65awSoNVwaoMVg2r/lrr9o9DLUAtDF3fwGr0sFpQraG3lrCWelgrUCud'
            'WoNaG1q4gbUxWFtY2w810RkPO1A7Q1h7WPu/tO6V/6Lr/alu+NRd97L0FG9Vd6B2hrD2sCxN7GH1Wtff6h5hHQ1xnWCd9K4/'
            'gzobwhpgDXpYI6jxq8P6QtdfgF30NhYr1ypWchuLDlSnU2tQa0MLt7C2hq4vMLMLw8wuMLMLw8wuMLOLXu/7A6iDTmGNKAxr'
            'RIE1ojgZLCwSxVlvItaIQl8jCqwRxWhoIZaI4r8tERQd7ArsasDKmYuVM72VpQfL+5jlP1s+LN9gBbACeVCUIahQ764Kq2q1'
            'NqyEFbZL1UbfS1RYoqutoZFYoaudtFO9VcUCXRkW6AoLdNUbLKzQlb5CV1ihq6NlSGCJrgxLdIUlujLs4yqs0ZW+RldYoyvD'
            'Gl1hja4M27gDnuPhKDfxgKd40DfjBzzEg+EhHvAQD4PBwlM8jHoT8RAPF30ZPCI1fv1F3J/COiIzfv093J8pJMajr3f8EYnx'
            'qCfGIxLjMdSpCFSkUzGoWKfmoOYydSlc6lLoVAmq1KkKVKVTNahapxagFjrVgGp0qgXV6tQS1FKncHlw0S8PLrg8uOiXBxds'
            'cy9rnUJavchp1avdXDiV1X2pVw+gBp0aQY06dQV1lamFmwinsk55oDydCkGF8gnRW8SwYjUsP0odairrVAYq06kcVK5TBahC'
            'p0pQpU5VoCqdqkHVKhWs3L6ayvIYDVYlLLmzgtE9sU7lDx4zHewI7KjHdQJ10qkzqLNODaAGnRpBjTp1AXWxPMMrMDlZBBc3'
            'WUxl7bPqqW7Yuz0/lfU2hv0IbLRgF2CG3g8PbpdNZQvmAfP0/j/4sHyDFcAKNGv+duP9tBTeyx9bVsNnq4L1scQBygN1+PmH'
            'z//42/Hzr7/+/ulvm59/+fH7H/7T9fwXmrpAeAtDUxtYjd7UBk1tXtDUJcJb6k0tU9cqU7mpE+WBMje1zBBeJh1kblVzUB/b'
            '2cUu5YGyt7RAeB/cAcWu5cGS4vPeTS9wl5KprHafF1SgPraSJC7lgbK3tEZ4tTol3igPlBRe8GwuEN4H33GLXcuDZe++BvE1'
            'H0zSTx3ox+5QmcrqqPPjGpT8WP14AWoh7QJvVRtQjU61oFq9gUtQH3x9+NlauQuHLx7HpqpBe3GoqSzu/26Vr8Cu2pHzre5y'
            '5lrLmcHyYHm61btjfiqrfR/6kUNNZXlbGvoxrFh/juHB3SJMZRvmAfvLBfF5Lfzzv9GzmiPE3BJiAey/XCv9p13HG+YBe0V7'
            'S4RY6mNlsjxYYoThs1ohwsoQYYUIKzXC+FmtEWGtj5kodh/IVJabG13PrnU9G6wB1mCwRlij9on2VDc+u22cygZrgKW3MT6P'
            'sCxtvMC6aHEl0wEjfnqp6l7WPmu/1d3CUt97vVXeAdsZAtvD2lsC64H1hsCOsI6WwE7ATobARlijwbrAuhisK6yrbs0j15pr'
            'L2HcqsagYkNYKazUYOWwcsP4mhfAiq9/KfcLWgWtMjSzhlUbrAWshaXLWmCtIbAlrKV4Or9V7oB1llZi3Z9vDa3Esj83LPtz'
            'rNRzw0o9P8A6GCws1POTpfMHYIMhMKyuc8PqmniulXgGy4flG6wAVmCwQlihwUI2SiKDhXSUGNJRksHKDBbSUWJJRwnSUVJY'
            'MGSQpLZgSCHJwtBlDazGYCEbJa2lkUhHydIQGLJR0hksJKPEkIwSJKPEkIwSJKOkt3Q+slFysGA4hCRHQytxbkgM54YE2Sgx'
            'ZKN05lrpzNBhKVJbakhtKVJb6lsCQ25LAwuG5JYakluK5JYakluK5JbGlkbOgc0tGE5uqeHkliLtpoa0myLtprnBQtZNC4NV'
            'wiotnY/zZGo4T6ZI4KkhgadIuqnhCJiuYK0M1hrW2mDhEjHdWB4kLutSy2VdiqybGo6AKfJkarmsS8/AzhYMZ8DUcAZMkcFT'
            'QwZPkcFTQwbPkHQzz9BhGbJuZsm6GRJlZkiUGZJbpn6+equM5JbNDYEht2WppceQkDLLOTBDFsnULFJ8m/nBU3q7l6WP3W9V'
            'K1CVTi1ALcRRcavcAGu0T48elgfL9BbSm5Enbnx5IvdbvgK10qk1qLVM1blL1bmF8kCZ+7/GuK0r7S2WW90aVq03tUZT6xc0'
            'FfOqXujhLRDeR97He/7XQTF+65U+VWsM4HpteKxrNHZteznyZmwR31Zv6x5L3L7RXoK71W1htXq/7ZewlrIV5O4qMpX1nBBU'
            'bodNZQPWbFys2aizKtiWLrUt5QcZ7Heutd/JYe33oPZ6WEd3Xk5lfUwc0fPHjW4NkWsNkdxdQwwqNoQ1hzXXrYs7tYNLKy85'
            'wWUJa6l2V3h1E/BUlsOKMLGjqjFYLayv6a6/SG3R899n3st6YM9/U3kvy1bsnx1rKhusEdZosC6wLrrVoo2toY3tAGswWOiv'
            'drRZsWutP/32/a9v3x7+9w/tEuONuzOZynJY87ObtaeywSpgFbKVbN3kOJUN1h7W3mD1sHqDdYAlLjmbb7NZ8DRU72Vxw3Sr'
            'fAF2EW8w3iqHnouF4odTt7oxrNgSWAos1TL3rW4GKzNYBazC8CjDElipB5YtXStb6k/y6dv7H2WD1cHqDNYa1tpg9bB6g3WA'
            'dTBYR1hH3aoxvurSYFWwvrjP/M9/9zB79mp4tSG2BayFMTbMpdowl2rMpdowl2rMpdowl+oNrI1hta4xAWrDBKgxAWrLBDjD'
            'OhusAdZgsLAhqEeDhf1AfTFYV1hX3Xr+kqx7WfxW5rfKI0b+2Gl/EHOri5E/aj+kcKu6BbXVqR2onU4htY29vnUdMbPHgx4W'
            'JvZ41KkTqJNOYYkYz4bOwhIxal/ku33b1T8N+HtZm4m3umtYaz2sDaiNTm1BbQ0t3MHaGawelvrq5a3yEdjRgp2AnfTOP4MS'
            '//Dxre7Sd62lr3f+MoAVGKwQVmjo+2UELDIEFsOKDdYc1txgJbASg5XCSg1WBiuzPMgcWG4IrIBVGKwSVmmwKliVwaph1QZr'
            'AWshL2DLFlRrCGsJa2mwkLeXhry9RN5ermXL893FcCqrXe/5EahIzh2eH8OK9bASUIlOpaBSncpAZTqVg8p1qgBViGezW+US'
            'WKnHVYGqdKoGVevUAtRCpxpQjUyt3P3zVJY+Q75V3YHaaffQt7p7WHuD1cPq9SYeQB0MYR1haV/wu3+78X8aD/eytpbe6i5h'
            'LcVt0q1yB0zMY291L5FrXSK5kV7jO9ZU/lDfe89UACpQqej5tznvZfFToVvlLTDxE+Sp7vw0d6yprFn92w98PK0S97K0dt2q'
            '7kHtdaoH1evUAdRBp46gjjp1AnX66u8B+7N1hnXWwxpADTo1ghp16gJKvAy/1b3CusphnWYudZrplAfK0ykflK9TAahAp0JQ'
            'oUp5WeNQU1lK1LeqLahWSha3qktQSz2qFaiVTnWgOp1agxJfUr3V3cDayAnMy7awtnoTd6B2OrUHtdepHlRv6KwDrIMe1hHU'
            '8WMZzLFOsE56WGdQZ50aQA06NYIadeoC6qJSfuiupH7YyCPLD1ewVnpYHahOp9ag1jq1AbXRqS2orU7tQO10ag9qr1M9qF6n'
            'DqAOOnUEddSpE6iTTp1BnXVqADXo1Ahq1KkrqKtMRe6OeyrrlAfK0ykflK9TIahQpyJQkU7FoGKZ6iqX6iqdQvbqGp1aglrK'
            'e1u/QyLs9ETYIRF2eiLskAg7PRF2SISdngg7JMJOT4QdslenZ68O2auTs1e0d7t9Kss7rGi/gbUxWDtYO4O1h6X9rUcef5vN'
            '2vdV+VGWLpXvdX1Y2hss97oRrEi7hb9XjoHF2ud098oJsMSCpcBSQ5dlsDJLl+XAckNgBazCElgFrLJgC2ALQysbWI1hJi1h'
            'LQ1xrWCtDNYa1trS+Rtg4qc998pbYFtDK3ewdpZW7oHt9cB6jP3eMvZ7jP3eMPZ7jP1e+jz+XrUF1Vra2AHr9Lgw9Pu1TmHg'
            '9xudwrDvtzqFUd/vdApjvt/r1AHUwTIcjsCOelwnUCedOoM6y9SA3hoOeg4a0FnD0WChicPZYA2wBoM1whr1rr+AuhjCusK6'
            '6tY4c61xJjdxxFlh9HQqBBUaWoijwhjpx6FxDmsux+Wt3G2vtyr0/Y23qoHVFqwF1lqwDbCN3P1+666qU1m5CHirGlzdqR1c'
            'P/plAd47Fs7cSTSVLZgH7INHbsfyYfmWwBJgiQVLgaWGVuawcoNVwCo0q3n7o/Kn4XovSwvig/JAWX4j+W6cEd5ZD28ANYjb'
            'sFvlC7CLHtcF3XZ5QbddEd5VD++K8K728PKZG14+k8PLPVCeeJV1q+wD8//btwf8xer9wDxgH/iRyKf/PHtGA0QY6D0XIL7A'
            '9MWYdyNEeNpXGt/rRrAik+XBsrc1Rnyx/ijmoOYWygNln64Jwks++PuzgYt5wOyPIkWAqWGoZLAy/VlkaGr2gmdRILxCD69A'
            'eMULnkSJ8EpDks1rYLXe1hptrcWv2XXQBeJbaCetW90lrKXe1iXaunzBsOsQXmdoaof4OttXHt+RNQJc6/23RnzrF/TfFuFt'
            '9fC2CG/7iu7bIb6dHt8e1N5CeaDsT6JHeL0hVxxgHfSmHtDUwwuaekR4Rz28I8I7viA8nO1y/Ww3UR4oeyrDeTEf9PAGhDeY'
            'vvf8buCIl+tHvBxHvPxq770CR7xCP+JNlAfKPPYKHBsLTw8PR7IisFAeKPM4KXC2KPSzRYnNXVlaKA+U1NL42awQXqXvUSbL'
            'g6Ul2ec5UWK/WC70/lsgvoV9ypYNwmv08BqE17wgvBbhtXp4LcJrxacbPaMrxLfS41shvpV9xSuxgy87PTxs4MvuBeFhg1xu'
            'DZMXO+Rya19GS2yQS32DPFEeKHv3YdNd7g3dh113+YJdd4mdcnnQd90ltrWlvq0tsa0tjy8YKdjWlvq2tsS2tjy/4EmMCG80'
            'jBTsQcuryfJgmdtaYRNa6ZvQCpvQ6gWb0Aqb0ErfhE6UB8oeHj4NqXx9zla4LK9ig4UdbTU3WLg5rlL9EeDiuNIvjqscVG5o'
            'IXbFVWWwsAWr9C3YAh+gLi56WAssQ4ur4dK5wZrRzMQXS26VcTb8+i+W+UJg+DysieTOb/BxWPOCj8MazPDGMMMbzPBGP7M2'
            '+DysecHnYQ0WjUZfNCbKA2UPDwtRoy9EDT7BarIXDBQsbk1uma34iKipv/7j/y9oWN8a7Ye97nVxHGz042CD42DzguNgg+Ng'
            '0xnem3j65b5HWfpiontdHASbrWV44NTW6Ke2BiesZm+JC29dNSfDYol76mYwWDgcNKPBQoZvDBm+R1z9KD/HASl5MKTkAa+7'
            'DKEeFrL7oGf3AdlzmBtaiFwyZLLlJe5aPZWlN2MflAfK/BmolywQ30KPrwHV6FQLSs9JXrKEtdTDWuIJLM03J16yQngr+b3W'
            'N8uDZc2ZXtIhvk7vvg7hiVeo4bO5RnhrPbw1wlu/4OluEN5GD2+D8DYv6L0dwtvp4e0Q3u4F4e0R3l4Pb4/w9i94uD3C6/Xw'
            'eoTXv6D3DgjvoId3QHiHF4R3RHhHPbwTqJOF8kDZl9Azwjsbstn53z/V+of1dT/V+vw9jH9IHqKSWvr0caBXYbmrtN/U/sPy'
            'YJmfRLVFfFt5oFRbhGf/vM2rMGOrg34Nd8NiFxNHSoXZXx3s75F6FaZ/pU//6oj4jvbVqcKSUulLSoUlpXrBklJhSanOenhY'
            'B6rzCwbygPAGPbwB4Q0veLgjwhv18C6gLhbKA2V+QcOr3Zv9qSzHV7ufBr5R5idR+wjP18PzEZ5vvrb16gjhRXp4EcKLxKfr'
            'dF+M+GI9vhjxxfZloE4QXqKHlyC85AVPN0d4ub5bmSwPlr37CsRX6N1XILziBeGVCO+DP9nluZYHy762VIiv0rfdb1bsWuJm'
            'qq7Q0uoFAxmXeLV+iVcvEN7iBQMFF4O1fjFYNwivsR9aalwQ1voFYY0Lwnr5gmGMC7hav4CrcVlWry2UB8o+jHF6rPXLshqH'
            'x3rzgvBw3VPr1z01rnvq/gXDGCeW+mRY7nBkqU/mVcCfuS9++TNP7T5/5oPydSoAFVgoD5R1zPmzEOGFenghwgtfEF6C8BI9'
            'vAThJS8IL0V4qR5eivBSe3iJexs1lfXrHh935b5+V+53eKqd/FRD373lmcp6E0P/BExf4UL/DEu/k41a91W8qaxbS/e8PpVl'
            'K+7d5ziV5Y8W4z6FlcpWUrt3JlPZYF1hXXVr4fb9VJat3Hf7Pve1vi+St1/JfP+I+FFWXpO4V+1AdTq1BrXWqQ2ojU5tQW11'
            'agdqp1N7UHudOoI66tQJ1EmnzqDOOjWAGnRqBDXq1AXURaeuoK4yFcxcKpjplAfK0ykflK9TAahAp0JQoU5FoCKdmoOa61QC'
            'KpGppnKpRvrxkXvVGlStUwtQC51qQbU6tQS11Ckk+malU0j0TadTSPTNWqeQ6JuNTiHRN1uV8mM3eU1lnRpADTo1ghp16gLq'
            'olNXUFeZmrtr+1TWqRBUqFMRqEinElCJTmWgMp3KQeUqFazd5DWVdSoBJfdV0rj79qTRfrLnXncPa2+weli9bKW1Ow3Trzr2'
            'fuEd8PK2UXhfHh7lD/V9+ExdQUl721vVcOZS4UyOKvRAffCbmx3Lh+XrLQxABToVgYr0zopBSe8U3KvOQc3lBi4Tl1omOpWC'
            'SnUqA5VJbxDc6+awcrnflwWoQm9hCarUqQpUpVM1qFrvqwWohU41oBqdakG1el8tQS11agVqpTewA9Xp1BrUWqc2oDY6tQO1'
            '06kDqINOHUEd9cFwAnWSoxqRvcZIjmpE9hpjnUL2GvXsNSJ7jYmeJ0akrzHV+x3pa8x0CtlrzPXOQvYa9ew1InuNpaHfkXPG'
            'Wg8LOWfUc86I1X1cGlqI5X3Ul/cRy/vY6Z2F5X3Ul/dxC2qrU3tQe53qQfV6XyFTjAedQqYYj3oDkSlGQ6Y4gzrrDRxADTqF'
            'A/R40SkcoEf9AH3BAfoy0ykcoC+eTuH8fNHPzxecny/y+dmfu2eTqawOUX9eg6p1agFqoVMNqEanWlCtTq1ArXSqA9Xp1BrU'
            'Wqc2oDb6EN2B2ulR9aB6PaojqKNOnUGddWoAJa/twdU9m0zlD77BFj5jMbBYvogMr+6aPJWl6+S3uvPAzdBTWfrGk3vdAdZg'
            'sEZY2vd2Vv6UHWbvD/JRNlgxrFj8Bq577Tk06aB5r5qASnQqBZXqVAYq06kcVK5TBahCpypQ2vdN3uvWsGp9nIYb1wql+6x7'
            '1S2orU7tQO10ag9qr1M9KO3juXvdA6yDdCq/1z3COupNPIE6aW9j3yufgZ31uAZQg06NoEaduoC6WCgPlOmXFe7IFfFd5fii'
            'mUtFM53yQfk6FYAKLJQHyvIHTXcjRHihHh72JlGkU9iaRLFOYTMRJRbKA2X585m7gQ1KlOrhYYMSZfra/2bFrvXxPxW+18QG'
            'JSr0BpagSp3CXif62FWJByp2qa/rqi/0OnZNUa03cAFqoVMNqEanWlCthfJA2RfDJcJb6uGtQK0Mk3GFEbaSJ+MKnbZ6wQrW'
            'oaWd3mkdwuts39F0R9aIb63Ht0Z86xd0H44kkX4kiXAkifQjSYRzRNTrFI4R0UGncIqI9FNEhn1npu87c+w7c33f+fSb6I+y'
            'TmFjl+sbuxwbuzyyUB4o09f/3hHsFnN9t5hjt5IXFsoDZU5QOXZAub4DarEDais9QbXYt7S1yfJg2UdIi91Qq++G2gXiW5i+'
            'CO9uYIfVNob+wxarbU2WB+sFzwKbrHZpCBC7rNawy2qxj2n1fUyLG8t2YwgL+4N2a7I8WC94nth1tF/cdfzZePrPs2cO13vt'
            'SX4MHfJCp+eFDp9udHML5YEy54UOlxydfsnR4UKiS/Xx1uFGojPcSHT4zKTL9SbmeAK5fQnvkE67Sr8s75CtOj1bdcgsnSGz'
            'dA26rbGffjpkq86QrTpkq+4V2apDhulW+rNAgun0BNPhTNUdPvgbQMA8YNYFyd+5R4apLE8HfxfDiuUxEpTuJytBOcpxRbF7'
            'hIziqxxXNHfPkFPZYPmw/K/+TacvYAGwwBBYCCs0WDEsfVCkvbvlmMqalU6TZ/PUX/eyNMlvVUNQoYXyQFm+0vFuRAgv0sOL'
            'EF70gvBihBdrM/1hebBMK+TNSBBfondfgvAS27cy35EU8aV6fCniS1/weDOEl+nhZQgve8HTLRBeof3m3L1yCaw0DOUSjS1f'
            '8CwqxFfpz6JCeNUrhnKN+Go9vhrx1S/ovgXCW+jhLRDewvSVrHejQXiNHl6D8P7DCeKvHuwXpkeLAFt9euyvrrW/yo3tZy7V'
            'z3TKB+XrFPYnvb4/6bE/6fX9SY+9RK/vJXrk/T7WqTmouYXyQJnXjR4psNdTYJ+Dyi2UB8qcTXtk077QwysQXvGC8JCf+1IP'
            'D+m5L213LDcD6bnX03OPTNrXFsoDZX8QyKS9nkl7ZL2+sVAeKHtLkfP6Vg+vRXiteWnyfXcRnspqeL4/BzXXqQRUolMpqFSn'
            'MlCZTuWgcp0qQZU6VYGqdGoBaqFTDSh5gvuhu6fyw0CnQlChTkWgIp3CHAz1ORhiDob6HAwxB0N9DoaYg6E+B1N0e6p3+9NX'
            'pjzKOnUEdVSp4Ok7Kx9lmdq7T3Aqyye3YJ/Bkh9hMLpjNBjlMRo9/yXHvSz+scqt8gXYRb6njuqVa9Urg9XB6nTr5I6uqWyw'
            'drB2eufHjZvF4kZ8X2p4e6fh6U+F7mWDNcLS/lboVvUC6mII6wrrKoe19Fxq6emUD8rXFpxb3QBWoIcVgYp0KgYV689we3at'
            '7VkOa4sRvx30jt9ixG/1Eb/FiN9eDGFhxG/1Eb+budRuplOYPDt98uww4Hf6gN+FoEKdwtzZ6XNnh7mzM8yd3RzW3GAlsBK9'
            'iRmoTKdyULlOFaAKfRruSlilHlYFqtKpGlQtbkVulRfAFnpcDahGHqb+008LPspqWH59AHXQqTOos04NoAaVCip3Sk9lfTwE'
            'JzddTGV5+oS+u8hPZe2LG251Q1ih3sj4kDrYVJZHanzIYGUGK4eVG6wCVmGwSlilZC3aaaAX76P1UVYG/h+UB8ryAcfdSBGe'
            'do3wh+XBMsW3nozyKb57Weq+W9UMlDZy73VzWLkeVgGq0KkSVKlTNahapxpQjU61oFqdWoFa6VQHqtOpNai1Tm1AbXRqC2qr'
            'UsFmcKipLE/CYHOBpZ1Ab3WfvpH3Udby7b1yB0weEMG4AbWR2xhWM8eaynLfR7uzY01l6ect73VHWKNkNZtpaG7eu/5RVrr+'
            'VnWbu9RW250su+mxXd6H/aMsPcZ73RGWdIfTldPGJn5/e/tRlpr4VnceutY8NFhD5FpDZLBiWLFspfvasaayZk2PLXz6eZtH'
            '2WB5sDxt9bpX9oH5OhbNLg42leVWRpfOtS6dwVrDWhusDayNwdrC2hqsHaydbMW1a01lg7WHtTdYPaxetuaBO1bnwcVgXWFd'
            'dQsrxdywUswPiOtgiOuIuI56XEkdOdZUlq10HzjWVNas/dse6f0e5lHWFsP1YZqA3nuKfJQ1bLuaKu/fl+lH+WOtfLynvn1b'
            'Gcr3HPkoSz12rzuHNTdYCazEYGWwMt16+qu3R9lgnWGdDdYAazBYI6zRYF1gXQzWFdZVtw4z1zrMZCuOG8eaygarhdUarCWs'
            'pcFawVoZrA5WZ7A2sDYGawtra7B2sHa6dUZcZ0Nc5z2svcHqYfXS6f2tbjovHGsqS3Htqrf1/d16lA1WCas0WBUs7WWo3dsu'
            'qXvPHY+ywRphjQbrAutisK6wrtKnaodgGpvX9/P7oyzFda+7gLUwWA2sRray2dWxprJueTPX8mYGy4PlGSwfli+PicwLYYWG'
            'uCJYkSGuOSxtH32a5kz29LMjj7JmnaY1OXh/v+BRlqzzfDoxPv269qMs5Y7h7de04/f5+ChLcQ3Hb7P86T7tUTZYISxtfI3j'
            '1Kb0/dPQR9lkebDEPygOn9UMEWqnrOs0GpKnv2x9lA1WDOvDM+D/A5WAyvyKdQEA'
        ),
    ),
    'slide_manifest.csv': ('1c87560ce01f49a78fa122ae3bd3117568304f35d5b5d0fb5a69a69abcac778d',
        (
            '...'
        ),
    ),
}

# Use only the embedded, hash-checked files, even if run from a checkout.
_starter_workspace = tempfile.TemporaryDirectory(prefix='cancer-ai-lab-')
DATA = pathlib.Path(_starter_workspace.name)
for name, (expected_hash, packed) in BUNDLE.items():
    data = gzip.decompress(base64.b64decode(packed, validate=True))
    if hashlib.sha256(data).hexdigest() != expected_hash:
        raise ValueError(f'Starter file checksum mismatch: {name}. Download a fresh copy.')
    (DATA / name).write_bytes(data)
del BUNDLE
subprocess.run([sys.executable, str(DATA / 'fetch.py'), '--no-rna'], check=True)
print('Verified the bundled starter files and the required Xena tables.')

## 2 · What does one row mean?
`sample_manifest.csv` has one row per **sample**. A patient can have more than one (a tumor and a normal-tissue sample, for example). The `has_*` columns say which data layers exist for that sample.

In [ ]:
import pandas as pd
samples = pd.read_csv(DATA / "sample_manifest.csv")
print(samples.sample_type.value_counts().to_string(), "\n")
samples.head()

## 3 · Every join changes the cohort
Start with primary tumors and add one data layer at a time.

In [ ]:
import matplotlib.pyplot as plt
steps, cohort = [], samples[samples.sample_type == "Primary Tumor"]
steps.append(("Primary tumors", len(cohort)))
for layer in ["has_survival", "has_rna", "has_copy_number", "has_mutation", "has_protein"]:
    cohort = cohort[cohort[layer]]
    steps.append(("+ " + layer.replace("has_", "").replace("_", " "), len(cohort)))
labels, counts = zip(*steps)
ax = pd.Series(counts, index=labels).plot.barh(color="#4C72B0", figsize=(8, 3.2))
ax.invert_yaxis(); ax.set_xlabel("Samples remaining"); ax.set_title("Each added data layer shrinks the cohort")
for i, n in enumerate(counts):
    ax.annotate(f" {n}", (n, i), va="center")
ax.set_xlim(0, max(counts) * 1.12); plt.tight_layout(); plt.show()

## 4 · Join clinical ER status to ER-alpha protein

In [ ]:
clinical = pd.read_csv(DATA / "raw/clinical.tsv", sep="\t", low_memory=False)
protein = pd.read_csv(DATA / "raw/protein_rppa.tsv.gz", sep="\t", index_col=0)
er = clinical.set_index("sampleID")["breast_carcinoma_estrogen_receptor_status"]
tumors = [s for s in protein.columns if s[13:15] == "01" and er.get(s) in ("Positive", "Negative")]
df = pd.DataFrame({"sample_id": tumors, "er_ihc": [er[s] for s in tumors],
                   "er_alpha_protein": protein.loc["ERALPHA", tumors].values})
df.groupby("er_ihc").er_alpha_protein.describe().round(2)

## 5 · Find the matching tissue slide
The output lists the pathology slide and its NCI Imaging Data Commons link. The external viewer may be unavailable; if it does not open, use the displayed slide ID and type as your evidence pointer and continue the exercise.

In [ ]:
slides = pd.read_csv(DATA / "slide_manifest.csv")
example = slides[(slides.sample_id == df.sample_id.iloc[0]) & (slides.slide_kind == "diagnostic (FFPE)")]
example[["slide_barcode", "slide_kind", "viewer_url"]]

## Your turn
Pick a product from [`products/`](../products/README.md). Some questions this data raises right away:
- Which tumors disagree between the clinical test and the protein measurement, and what else do we know about them?
- Do your conclusions hold if you change which samples are included?
- What happens to your tool when a new release changes some samples? Build one with `python data/tcga-brca/make_release.py`.